In [39]:
import os

if not os.path.exists("pw18-ticketing"):
    !git clone https://github.com/simonaoliveto83/pw18-ticketing.git

In [40]:
import pandas as pd
import joblib
import torch
import torch.nn as nn

In [41]:
BASE_PATH = "pw18-ticketing/prototipo2/models"

CATEGORY_MODEL_PATH = BASE_PATH + "/category_model.pt"
VECTORIZER_PATH = BASE_PATH + "/tfidf_vectorizer.joblib"
PRIORITY_MODEL_PATH = BASE_PATH + "/priority_model.pt"
VECTORIZER_PRIO_PATH = BASE_PATH + "/tfidf_prio_vectorizer.joblib"

In [42]:
class TicketClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [43]:
class TicketPriorityClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [44]:
# =========================
# 1. Caricamento modelli
# =========================
vectorizer = joblib.load(VECTORIZER_PATH)
cat_input_dim = len(vectorizer.get_feature_names_out())
vectorizerPrio = joblib.load(VECTORIZER_PRIO_PATH)
prio_input_dim = len(vectorizerPrio.get_feature_names_out())

cat_model = TicketClassifier(cat_input_dim, 3)
prio_model = TicketPriorityClassifier(prio_input_dim, 3)

cat_model.load_state_dict(torch.load(CATEGORY_MODEL_PATH, map_location="cpu"))
prio_model.load_state_dict(torch.load(PRIORITY_MODEL_PATH, map_location="cpu"))

cat_model.eval()
prio_model.eval()

TicketPriorityClassifier(
  (net): Sequential(
    (0): Linear(in_features=1358, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
)

In [45]:
# =========================
# 2. Caricamento dataset
# =========================
from google.colab import drive
drive.mount("/content/drive")

df = pd.read_csv("/content/drive/MyDrive/export/tickets_input.csv", delimiter=";")

# Controllo colonne
required_cols = ["id", "title", "body"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Manca la colonna: {col}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
# =========================
# 3. Preprocessing
# =========================
# Unisco title e body (come fatto in training)
df["text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.lower()

In [47]:
# =========================
# 4. Trasformazione TF-IDF
# =========================
X = vectorizer.transform(df["text"])
X_tensor = torch.tensor(X.toarray(), dtype=torch.float32)

In [48]:
# =========================
# 5. Predizioni
# =========================
with torch.no_grad():
    cat_logits = cat_model(X_tensor)

    # Create a separate tensor for priority model using its specific vectorizer
    X_prio = vectorizerPrio.transform(df["text"])
    X_prio_tensor = torch.tensor(X_prio.toarray(), dtype=torch.float32)
    prio_logits = prio_model(X_prio_tensor)

    cat_pred = torch.argmax(cat_logits, dim=1).numpy()
    prio_pred = torch.argmax(prio_logits, dim=1).numpy()

In [49]:
# =========================
# 6. Mapping etichette
# =========================
cat_map = {0: "Amministrativo", 1: "Tecnico", 2: "Commerciale"}
prio_map = {0: "Bassa", 1: "Media", 2: "Alta"}

df["predicted_category"] = [cat_map[i] for i in cat_pred]
df["predicted_priority"] = [prio_map[i] for i in prio_pred]

In [50]:
# =========================
# 7. Export CSV
# =========================
from google.colab import drive
drive.mount("/content/drive")

df[["id", "text", "predicted_category", "predicted_priority"]] \
    .to_csv("/content/drive/MyDrive/export/predizioni_ticket_poc2.csv", index=False)

print("✅ File predizioni_ticket.csv creato con successo!")

✅ File predizioni_ticket.csv creato con successo!
